In [0]:
# spark.sql("DROP TABLE IF EXISTS dbr_dev.joshuandegwa_bronze.lab3_flights_raw")

In [0]:
dbutils.widgets.text("container", "")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("secret_scope", "")

In [0]:
# ==============================================================================
# Setup Credentials & Catalog Target
# ==============================================================================

# 1. Retrieve Parameters
_container = dbutils.widgets.get("container")
_storage_account = dbutils.widgets.get("storage_account")
_secret_scope = dbutils.widgets.get("secret_scope")

# 2. Azure Storage OAuth Authentication
spark.conf.set(
    f"fs.azure.account.auth.type.{_storage_account}.dfs.core.windows.net",
    "OAuth",
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{_storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{_storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(
        scope=_secret_scope, key="sp-databricks-adls-appid"
    ),
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{_storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(
        scope=_secret_scope, key="sp-databricks-adls-appkey"
    ),
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{_storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{dbutils.secrets.get(scope=_secret_scope, key='tenant-id')}/oauth2/token",
)

# 3. Path Definitions
BASE_PATH = f"abfss://{_container}@{_storage_account}.dfs.core.windows.net"
LANDING_PATH = f"{BASE_PATH}/lab3/landing_zone/"
CHECKPOINT_PATH = f"{BASE_PATH}/lab3/_checkpoints/bronze_flights/"

# 4. Set Active Database Schema to your Bronze database
TARGET_SCHEMA = "joshuandegwa_bronze"
spark.sql(f"USE DATABASE {TARGET_SCHEMA}")

print(f"✅ Ready to ingest from {LANDING_PATH}")
print(f"✅ Target database set to: {TARGET_SCHEMA}")

In [0]:
# ==============================================================================
# Auto Loader Ingestion into Catalog Table
# ==============================================================================

from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    StringType,
    StructField,
    StructType,
)

# 1. Define explicit schema matching raw telemetry files
flight_schema = StructType(
    [
        StructField("icao24", StringType(), True),
        StructField("callsign", StringType(), True),
        StructField("origin_country", StringType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("latitude", DoubleType(), True),
        StructField("baro_altitude", DoubleType(), True),
        StructField("velocity", DoubleType(), True),
        StructField("true_track", DoubleType(), True),
        StructField("vertical_rate", DoubleType(), True),
        StructField("on_ground", BooleanType(), True),
    ]
)

# 2. Configure Auto Loader stream reader
raw_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(flight_schema)
    .load(LANDING_PATH)
)

# 3. Write stream directly to your Catalog table: joshuandegwa_bronze.lab3_flights_raw
query = (
    raw_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)  # Ingests all available files and finishes
    .toTable("joshuandegwa_bronze.lab3_flights_raw")
)

query.awaitTermination()

print(
    f"✅ Ingestion complete! Table 'joshuandegwa_bronze.lab3_flights_raw' successfully created."
)

# 3. Print Streaming Statistics
progress_history = query.recentProgress

if progress_history:
    total_rows = sum(p.get("numInputRows") or 0 for p in progress_history)
    batch_count = len(progress_history)
    
    # Grab execution timing from the active batches
    total_duration = sum(p.get("durationMs", {}).get("triggerExecution", 0) for p in progress_history)
    avg_rate = (total_rows / (total_duration / 1000.0)) if total_duration > 0 else 0.0

    print("=" * 60)
    print("      STREAMING BATCH METRICS REPORT")
    print("=" * 60)
    print(f"Total Micro-Batches    : {batch_count}")
    print(f"Total Rows Processed   : {total_rows}")
    print(f"Effective Processing   : {avg_rate:.2f} rows/sec")
    print(f"Total Execution Time   : {total_duration} ms")
    print("=" * 60)
else:
    print("ℹ️ No streaming progress history recorded.")

In [0]:
# Before new columns added. 
display(spark.sql("SELECT count(*) FROM joshuandegwa_bronze.lab3_flights_raw"))

In [0]:
%sql
-- 1. Check updated table schema (Notice new columns 'aircraft_type', 'passenger_count', and '_rescued_data')
DESCRIBE joshuandegwa_bronze.lab3_flights_raw;

In [0]:
%sql
SELECT COUNT(*) FROM joshuandegwa_bronze.lab3_flights_raw;

In [0]:
# ==============================================================================
# EXPERIMENT CELL: Trigger Type Comparison
# ==============================================================================

# Choose mode to test: "availableNow" | "once" | "continuous"
TRIGGER_MODE = "availableNow" 

stream_reader = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.schemaLocation", SCHEMA_EVO_LOCATION)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.allowOverwrites", "true")
    .load(LANDING_PATH)
)

writer = (
    stream_reader.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_EVO_PATH)
    .option("mergeSchema", "true")
)

# ------------------------------------------------------------------------------
# TRIGGER MODE 1: availableNow
# WHAT HAPPENS: Auto Loader scans the landing zone for all unconsumed files,
# processes them incrementally across multiple micro-batches until the backlog
# reaches 0, and then shuts down automatically. Best for cost-efficient batch jobs.
# ------------------------------------------------------------------------------
if TRIGGER_MODE == "availableNow":
    print("🚀 Running with trigger(availableNow=True)... [Drains all backlog files, then stops]")
    q_exp = writer.trigger(availableNow=True).toTable("dbr_dev.joshuandegwa_bronze.lab3_flights_evolved")
    q_exp.awaitTermination()

# ------------------------------------------------------------------------------
# TRIGGER MODE 2: once
# WHAT HAPPENS: Auto Loader reads ONLY a single micro-batch of newly arrived files 
# and stops immediately, even if more files remain in the landing directory.
# (Legacy option; availableNow is preferred in modern Databricks).
# ------------------------------------------------------------------------------
elif TRIGGER_MODE == "once":
    print("🚀 Running with trigger(once=True)... [Processes 1 micro-batch only, then stops]")
    q_exp = writer.trigger(once=True).toTable("dbr_dev.joshuandegwa_bronze.lab3_flights_evolved")
    q_exp.awaitTermination()

# ------------------------------------------------------------------------------
# TRIGGER MODE 3: processingTime (Continuous)
# WHAT HAPPENS: The stream stays active indefinitely in memory, waking up at 
# fixed intervals (e.g., every 10 seconds) to check for new files and ingest 
# them in near real-time. Must be manually stopped (query.stop()).
# ------------------------------------------------------------------------------
elif TRIGGER_MODE == "continuous":
    import time
    print("⚡ Starting Continuous Stream (10s interval)... [Runs continuously until stopped]")
    q_exp = writer.trigger(processingTime="10 seconds").toTable("dbr_dev.joshuandegwa_bronze.lab3_flights_evolved")
    time.sleep(30)  # Runs continuous polling for 30 seconds for test purposes
    q_exp.stop()
    print("🛑 Continuous stream stopped.")

print(f"✅ Trigger experiment for '{TRIGGER_MODE}' complete!")